# TP de Regresión — Predicción de precios de casas (Boston)
### Aprendizaje Automático 1 — Tecnicatura en Inteligencia Artificial (FCEIA - UNR)

**Integrantes:** Apellido1, Apellido2, Apellido3

---
Este notebook cubre, en esta primera entrega, el **punto 3 (Análisis Exploratorio de Datos)** de la consigna del TP. Las secciones 4 a 7 (modelado, regularización, hiperparámetros, comparación y conclusión) se agregan más adelante en este mismo notebook, respetando el orden.


---
## 1. Configuración del entorno y carga de datos

In [ ]:
# Librerías estándar para EDA y preprocesamiento
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

# Configuración global de visualización (consistencia entre gráficos)
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 100
})
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42  # semilla fija para reproducibilidad en todo el notebook

In [ ]:
# Carga del dataset
df = pd.read_csv('house-prices-tp.csv')  # ajustar el path si hace falta

print(f'Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
df.head()

---
## 2. Inspección estructural del dataset

In [ ]:
# Tipos de datos y presencia de nulos por columna (vista rápida)
df.info()

In [ ]:
# Separación de variables por tipo.
# Nota: CHAS es una dummy (0/1) del río Charles: aunque quede tipada como numérica,
# conceptualmente es categórica y hay que tratarla como tal en el análisis.
cols_numericas = df.select_dtypes(include=[np.number]).columns.tolist()
cols_numericas.remove('MEDV')  # el target se analiza aparte
col_categorica = 'CHAS'
if col_categorica in cols_numericas:
    cols_numericas.remove(col_categorica)

print(f'Variables numéricas continuas ({len(cols_numericas)}): {cols_numericas}')
print(f'Variable categórica (dummy): {col_categorica}')
print('Variable target: MEDV')

---
## 3. Análisis descriptivo

Medidas de tendencia central, dispersión, sesgo y curtosis para cada variable numérica.
Esto permite entender rango de variación y forma de la distribución de cada feature antes de decidir
transformaciones o tratamiento de outliers.

In [ ]:
desc = df[cols_numericas + ['MEDV']].describe().T
desc['mediana'] = df[cols_numericas + ['MEDV']].median()
desc['sesgo'] = df[cols_numericas + ['MEDV']].skew()
desc['curtosis'] = df[cols_numericas + ['MEDV']].kurtosis()

desc[['mean', 'mediana', 'std', 'min', '25%', '50%', '75%', 'max', 'sesgo', 'curtosis']]

**TODO:** interpretar la tabla anterior. ¿Qué variables presentan sesgo marcado (|sesgo| > 1)?
¿Alguna sugiere una transformación (log, sqrt) más adelante? Redactar 2-3 líneas de conclusión acá.

In [ ]:
# Distribución de la variable dummy CHAS
df[col_categorica].value_counts(dropna=False)

---
## 4. Análisis y decisión sobre datos faltantes

La consigna pide explícitamente detectar nulos y **justificar** la decisión de tratamiento
(eliminar filas, imputar, o dejarlos si el modelo lo tolera). No hay una única respuesta correcta:
lo que se evalúa es el razonamiento.

In [ ]:
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df)) * 100

reporte_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': pct_nulos.round(2)})
reporte_nulos = reporte_nulos[reporte_nulos['Nulos'] > 0].sort_values('Nulos', ascending=False)
reporte_nulos

In [ ]:
# Visualización del patrón de nulos (ayuda a ver si están concentrados en las mismas filas o dispersos)
plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Patrón de valores faltantes')
plt.show()

**TODO — decisión a justificar:**
- ¿Los nulos parecen MCAR, MAR o MNAR? ¿Por qué?
- ¿Se imputan (media/mediana para numéricas, moda para `CHAS`) o se eliminan las filas?
  Con ~3-5% de nulos por columna, eliminar todas las filas con al menos un nulo puede perder
  una fracción grande del dataset (chequear cuántas filas quedarían) — comparar ambas opciones.
- Justificar la elección final antes de aplicarla en la celda siguiente.

In [ ]:
# Aplicar la decisión de imputación/eliminación tomada arriba.
# Ejemplo de esqueleto (ajustar según lo decidido y lo justificado):

# Opción A: imputar numéricas con mediana (robusta a outliers) y CHAS con la moda
# for col in cols_numericas + ['MEDV']:
#     df[col] = df[col].fillna(df[col].median())
# df[col_categorica] = df[col_categorica].fillna(df[col_categorica].mode()[0])

# Opción B: eliminar filas con nulos
# df = df.dropna()

# Verificación final: no deberían quedar nulos
print('Nulos restantes:', df.isnull().sum().sum())

---
## 5. Visualización de datos

In [ ]:
# Histogramas de todas las variables numéricas (incluye el target MEDV)
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for i, col in enumerate(cols_numericas + ['MEDV']):
    ax = axes[i]
    sns.histplot(df[col], kde=True, ax=ax, color='steelblue', edgecolor='white', alpha=0.7)
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1, label='Media')
    ax.axvline(df[col].median(), color='green', linestyle='--', linewidth=1, label='Mediana')
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)

for j in range(len(cols_numericas) + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de variables numéricas', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots para detectar outliers a simple vista
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for i, col in enumerate(cols_numericas + ['MEDV']):
    ax = axes[i]
    sns.boxplot(y=df[col], ax=ax, color='steelblue')
    ax.set_title(col, fontweight='bold')

for j in range(len(cols_numericas) + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplots de variables numéricas', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Scatterplots de cada feature contra el target MEDV
# (clave en un problema de regresión: da una primera idea de qué variables
#  tienen relación lineal fuerte con el precio)
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for i, col in enumerate(cols_numericas):
    ax = axes[i]
    ax.scatter(df[col], df['MEDV'], alpha=0.4, s=15, color='steelblue')
    ax.set_xlabel(col)
    ax.set_ylabel('MEDV')

for j in range(len(cols_numericas), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Relación de cada variable con el target (MEDV)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# MEDV según la variable dummy CHAS (¿limitar con el río impacta en el precio?)
plt.figure(figsize=(6, 5))
sns.boxplot(data=df, x=col_categorica, y='MEDV', palette='viridis')
plt.title('MEDV según CHAS (limita con el río Charles)')
plt.show()

**TODO:** comentar qué variables muestran relación aparente (lineal o no) con `MEDV`
y cuáles no parecen aportar mucha información a simple vista.

---
## 6. Codificación de variables categóricas

`CHAS` es la única variable categórica del dataset y ya viene codificada como dummy (0/1),
por lo tanto no requiere un encoding adicional (no hace falta one-hot ni label encoding).
Si en el análisis se decide binear alguna variable continua en categorías, el encoding
correspondiente se aplicaría acá y debe justificarse.

---
## 7. Matriz de correlación

In [ ]:
corr = df.corr(method='pearson')

mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(11, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'label': 'Coeficiente de Pearson'})
plt.title('Matriz de correlación de Pearson', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Variables más correlacionadas con el target
corr['MEDV'].drop('MEDV').sort_values(key=abs, ascending=False)

In [ ]:
# Pares de features con posible colinealidad (correlación fuerte entre sí, no con el target)
UMBRAL_COLINEALIDAD = 0.8
features = [c for c in corr.columns if c != 'MEDV']

pares_colineales = []
for i in range(len(features)):
    for j in range(i + 1, len(features)):
        r = corr.loc[features[i], features[j]]
        if abs(r) >= UMBRAL_COLINEALIDAD:
            pares_colineales.append((features[i], features[j], round(r, 3)))

pares_colineales

**TODO:** interpretar la matriz — ¿qué variables predictoras se relacionan más con `MEDV`?
¿Hay pares de features muy correlacionados entre sí (colinealidad) que puedan afectar
a la regresión lineal más adelante (ej. inflar los coeficientes)?

---
## 8. Detección de outliers (método IQR)

In [ ]:
IQR_FACTOR = 1.5
reporte_outliers = []

for col in cols_numericas + ['MEDV']:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - IQR_FACTOR * iqr, q3 + IQR_FACTOR * iqr
    n_outliers = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
    reporte_outliers.append({
        'Variable': col, 'Lim_inf': round(lim_inf, 2), 'Lim_sup': round(lim_sup, 2),
        'N_outliers': n_outliers, 'Pct_outliers': round(n_outliers / len(df) * 100, 2)
    })

pd.DataFrame(reporte_outliers).sort_values('N_outliers', ascending=False)

**TODO — decisión a justificar:** ¿se recortan/capean los outliers, se eliminan las filas,
o se dejan (por ejemplo si son valores plausibles y el modelo de regularización puede manejarlos)?
Justificar la decisión final antes de avanzar al split.

---
## 9. División train / test

Se hace el split **antes** de estandarizar, para evitar *data leakage*: el escalador se ajusta
(`fit`) solo con los datos de entrenamiento y luego se aplica (`transform`) a train y test por igual.

In [ ]:
X = df[cols_numericas + [col_categorica]]
y = df['MEDV']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]} filas | Test: {X_test.shape[0]} filas')

---
## 10. Estandarización de variables numéricas

Necesaria porque los métodos de gradiente descendente y de regularización (Ridge/Lasso/Elastic Net)
son sensibles a la escala de las features — sin estandarizar, variables como `TAX` (rango en cientos)
dominarían sobre variables como `RM` (rango ~3-9) solo por magnitud.

In [ ]:
scaler = StandardScaler()

# Fit SOLO sobre train, transform sobre train y test
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)

X_train_scaled.describe().T[['mean', 'std']]  # verificación: media ~0, std ~1 en train

---
## Próximos pasos

Con `X_train_scaled`, `X_test_scaled`, `y_train`, `y_test` listos, en la siguiente sección del
notebook (punto 4 de la consigna) se implementan `LinearRegression`, gradiente descendente
(`SGDRegressor`) y los modelos regularizados (`Ridge`, `Lasso`, `ElasticNet`).